# Phase 4 : Validation avec Great Expectations

Suite d'expectations automatisées couvrant les 6 piliers :
- **Complétude** : colonnes non nulles, taux minimum
- **Exactitude** : types de données
- **Validité** : plages autorisées  
- **Cohérence** : comparaisons inter-colonnes
- **Unicité** : clés primaires
- **Actualité** : dates récentes, pas de futures

In [ ]:
import great_expectations as gx
import pandas as pd
import os
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv('../.env')  

DB_HOST = os.getenv('DB_HOST', 'host.docker.internal')
DB_PORT = os.getenv('DB_PORT', '5432')
DB_NAME = os.getenv('POSTGRES_DB', 'games_db')
DB_USER = os.getenv('POSTGRES_USER', 'postgres')
DB_PASSWORD = os.getenv('POSTGRES_PASSWORD', 'postgres')

CONNECTION_STRING = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

try:
    engine = create_engine(CONNECTION_STRING)
    df = pd.read_sql("SELECT * FROM games_gold", engine)
    print(f"Données chargées depuis PostgreSQL: {len(df):,} lignes × {len(df.columns)} colonnes")
except Exception as e:
    print(f"Erreur de connexion PostgreSQL: {e}")
    # Fallback vers fichier CSV si base indisponible
    df = pd.read_csv('../data/Gold/games_gold.csv')
    print(f"Utilisation du fichier CSV en fallback: {len(df):,} lignes")

# Créer un contexte pour gx
context = gx.get_context(mode="file", project_root_dir="..")

/Users/U1094723/Documents/documents perso/cours/data governance/.venv/lib/python3.9/site-packages/great_expectations/data_context/store/_store_backend.py:88: DeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parsed_store_backend_id = store_backend_id_file_parser.parseString(


In [20]:
# Configuration de great expectations

datasource_name = "games_datasource"
asset_name = "games_gold"
suite_name = "games_quality_suite"

# Datasource
try:
    datasource = context.data_sources.get(datasource_name)
except:
    datasource = context.data_sources.add_pandas(datasource_name)

# Asset
try:
    data_asset = datasource.get_asset(asset_name)
except:
    data_asset = datasource.add_dataframe_asset(name=asset_name)

# Batch definition
try:
    batch_definition = data_asset.get_batch_definition("games_batch")
except:
    batch_definition = data_asset.add_batch_definition_whole_dataframe("games_batch")

# liste des règles à vérifier (vide pour l'instant)
try:
    suite = context.suites.get(suite_name)
    context.suites.delete(suite_name)
    suite = gx.ExpectationSuite(name=suite_name)
    suite = context.suites.add(suite)
except:
    suite = gx.ExpectationSuite(name=suite_name)
    suite = context.suites.add(suite)

batch = batch_definition.get_batch(batch_parameters={"dataframe": df})
# validator : c'est l'outil qui exécute les tests sur les données
validator = context.get_validator(batch=batch, expectation_suite=suite)

## 1. Complétude 

In [ ]:
# AppID ne doit jamais être null (clé primaire)
validator.expect_column_values_to_not_be_null("AppID")

# Name ne doit jamais être null
validator.expect_column_values_to_not_be_null("Name")

# Release date doit avoir au moins 95% de valeurs non nulles
validator.expect_column_values_to_not_be_null("Release date", mostly=0.95)

/Users/U1094723/Documents/documents perso/cours/data governance/.venv/lib/python3.9/site-packages/great_expectations/expectations/expectation.py:1604: UserWarning: `result_format` configured at the Validator-level will not be persisted. Please add the configuration to your Checkpoint config or checkpoint_run() method instead.
  warnings.warn(
Calculating Metrics: 100%|██████████| 6/6 [00:00<00:00, 1527.15it/s]
/Users/U1094723/Documents/documents perso/cours/data governance/.venv/lib/python3.9/site-packages/great_expectations/expectations/expectation.py:1604: UserWarning: `result_format` configured at the Validator-level will not be persisted. Please add the configuration to your Checkpoint config or checkpoint_run() method instead.
  warnings.warn(
Calculating Metrics: 100%|██████████| 6/6 [00:00<00:00, 1264.17it/s]
/Users/U1094723/Documents/documents perso/cours/data governance/.venv/lib/python3.9/site-packages/great_expectations/expectations/expectation.py:1604: UserWarning: `result_

{
  "success": true,
  "expectation_config": {
    "type": "expect_column_values_to_not_be_null",
    "kwargs": {
      "batch_id": "games_datasource-games_gold",
      "column": "Release date",
      "mostly": 0.95
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {
    "element_count": 122610,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": []
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

## 2. Unicité

In [ ]:
# AppID doit être unique
validator.expect_column_values_to_be_unique("AppID")

/Users/U1094723/Documents/documents perso/cours/data governance/.venv/lib/python3.9/site-packages/great_expectations/expectations/expectation.py:1604: UserWarning: `result_format` configured at the Validator-level will not be persisted. Please add the configuration to your Checkpoint config or checkpoint_run() method instead.
  warnings.warn(
Calculating Metrics: 100%|██████████| 8/8 [00:00<00:00, 546.19it/s] 


{
  "success": true,
  "expectation_config": {
    "type": "expect_column_values_to_be_unique",
    "kwargs": {
      "batch_id": "games_datasource-games_gold",
      "column": "AppID"
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {
    "element_count": 122610,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

## 3. Exactitude

In [ ]:
# Price doit être de type float
validator.expect_column_values_to_be_of_type("Price", "float64")

# AppID doit être de type entier
validator.expect_column_values_to_be_of_type("AppID", "int64")

/Users/U1094723/Documents/documents perso/cours/data governance/.venv/lib/python3.9/site-packages/great_expectations/expectations/expectation.py:1604: UserWarning: `result_format` configured at the Validator-level will not be persisted. Please add the configuration to your Checkpoint config or checkpoint_run() method instead.
  warnings.warn(
Calculating Metrics: 100%|██████████| 1/1 [00:00<00:00, 1189.20it/s]
/Users/U1094723/Documents/documents perso/cours/data governance/.venv/lib/python3.9/site-packages/great_expectations/expectations/expectation.py:1604: UserWarning: `result_format` configured at the Validator-level will not be persisted. Please add the configuration to your Checkpoint config or checkpoint_run() method instead.
  warnings.warn(
Calculating Metrics: 100%|██████████| 1/1 [00:00<00:00, 938.11it/s] 


{
  "success": true,
  "expectation_config": {
    "type": "expect_column_values_to_be_of_type",
    "kwargs": {
      "batch_id": "games_datasource-games_gold",
      "column": "AppID",
      "type_": "int64"
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {
    "observed_value": "int64"
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

## 4. Validité

In [ ]:
# Price doit être >= 0
validator.expect_column_values_to_be_between("Price", min_value=0, max_value=1000)

# Required age entre 0 et 21
validator.expect_column_values_to_be_between("Required age", min_value=0, max_value=21)

# Metacritic score entre 0 et 100
validator.expect_column_values_to_be_between("Metacritic score", min_value=0, max_value=100)

# Review score entre 0 et 100
validator.expect_column_values_to_be_between("Review score", min_value=0, max_value=100)

/Users/U1094723/Documents/documents perso/cours/data governance/.venv/lib/python3.9/site-packages/great_expectations/expectations/expectation.py:1604: UserWarning: `result_format` configured at the Validator-level will not be persisted. Please add the configuration to your Checkpoint config or checkpoint_run() method instead.
  warnings.warn(
Calculating Metrics: 100%|██████████| 8/8 [00:00<00:00, 484.79it/s] 
/Users/U1094723/Documents/documents perso/cours/data governance/.venv/lib/python3.9/site-packages/great_expectations/expectations/expectation.py:1604: UserWarning: `result_format` configured at the Validator-level will not be persisted. Please add the configuration to your Checkpoint config or checkpoint_run() method instead.
  warnings.warn(
Calculating Metrics: 100%|██████████| 8/8 [00:00<00:00, 636.38it/s] 
/Users/U1094723/Documents/documents perso/cours/data governance/.venv/lib/python3.9/site-packages/great_expectations/expectations/expectation.py:1604: UserWarning: `result_

{
  "success": true,
  "expectation_config": {
    "type": "expect_column_values_to_be_between",
    "kwargs": {
      "batch_id": "games_datasource-games_gold",
      "column": "Review score",
      "min_value": 0.0,
      "max_value": 100.0
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {
    "element_count": 122610,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

## 5. Cohérence 

In [ ]:
# Owners_max >= Owners_min
validator.expect_column_pair_values_A_to_be_greater_than_B(
    "Owners_max",
    "Owners_min",
    or_equal=True
)

# Positive reviews <= Total reviews
validator.expect_column_pair_values_A_to_be_greater_than_B(
    "Total reviews",
    "Positive",
    or_equal=True
)

/Users/U1094723/Documents/documents perso/cours/data governance/.venv/lib/python3.9/site-packages/great_expectations/expectations/expectation.py:1604: UserWarning: `result_format` configured at the Validator-level will not be persisted. Please add the configuration to your Checkpoint config or checkpoint_run() method instead.
  warnings.warn(
Calculating Metrics: 100%|██████████| 7/7 [00:00<00:00, 309.39it/s] 
/Users/U1094723/Documents/documents perso/cours/data governance/.venv/lib/python3.9/site-packages/great_expectations/expectations/expectation.py:1604: UserWarning: `result_format` configured at the Validator-level will not be persisted. Please add the configuration to your Checkpoint config or checkpoint_run() method instead.
  warnings.warn(
Calculating Metrics: 100%|██████████| 7/7 [00:00<00:00, 416.50it/s] 


{
  "success": true,
  "expectation_config": {
    "type": "expect_column_pair_values_a_to_be_greater_than_b",
    "kwargs": {
      "batch_id": "games_datasource-games_gold",
      "column_A": "Total reviews",
      "column_B": "Positive",
      "or_equal": true
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {
    "element_count": 122610,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

## 6. Actualité

In [ ]:
# Release year entre 1980 et 2026 (pas de dates futures)
validator.expect_column_values_to_be_between("Release year", min_value=1980, max_value=2026)

/Users/U1094723/Documents/documents perso/cours/data governance/.venv/lib/python3.9/site-packages/great_expectations/expectations/expectation.py:1604: UserWarning: `result_format` configured at the Validator-level will not be persisted. Please add the configuration to your Checkpoint config or checkpoint_run() method instead.
  warnings.warn(
Calculating Metrics: 100%|██████████| 8/8 [00:00<00:00, 577.24it/s] 


{
  "success": true,
  "expectation_config": {
    "type": "expect_column_values_to_be_between",
    "kwargs": {
      "batch_id": "games_datasource-games_gold",
      "column": "Release year",
      "min_value": 1980.0,
      "max_value": 2026.0
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {
    "element_count": 122610,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

## Exécution de la Validation

In [39]:
# Sauvegarde de la suite
suite = validator.get_expectation_suite()
suite.save()

# Création du ValidationDefinition et Checkpoint pour l'exécution automatique
validation_name = "games_validation"
try:
    validation_def = context.validation_definitions.get(validation_name)
except:
    validation_def = gx.ValidationDefinition(
        name=validation_name,
        data=batch_definition,
        suite=suite,
    )
    validation_def = context.validation_definitions.add(validation_def)

checkpoint_name = "games_checkpoint"
try:
    checkpoint = context.checkpoints.get(checkpoint_name)
except:
    checkpoint = gx.Checkpoint(
        name=checkpoint_name,
        validation_definitions=[validation_def],
        # actions sur échec : stockage des résultats + mise à jour Data Docs
        # on pourrait aussi ajouter SlackNotificationAction pour les alertes
    )
    checkpoint = context.checkpoints.add(checkpoint)

# Exécution
checkpoint_result = checkpoint.run(batch_parameters={"dataframe": df})

# Mise à jour des Data Docs (rapport HTML interactif)
context.build_data_docs()
context.open_data_docs()

print(f"Validation exécutée : {len(suite.expectations)} expectations")

Calculating Metrics: 100%|██████████| 31/31 [00:00<00:00, 516.93it/s]


Validation exécutée : 4 expectations


In [35]:
# résultats du checkpoint
results = list(checkpoint_result.run_results.values())[0]
stats = results.statistics
result_list = results.results


print(f"Expectations évaluées: {stats['evaluated_expectations']}")
print(f"Succès: {stats['successful_expectations']}")
print(f"Échecs: {stats['unsuccessful_expectations']}")
print(f"Taux de succès: {stats['success_percent']:.1f}%")

print("\nDétails :")
for result in result_list:
    status = "OK" if result.success else "FAILED"
    exp_type = result.expectation_config.type.replace("expect_column_", "").replace("expect_", "")
    kwargs = result.expectation_config.kwargs
    column = kwargs.get('column', kwargs.get('column_A', 'N/A'))
    print(f"   {status} {column}: {exp_type}")

Expectations évaluées: 4
Succès: 4
Échecs: 0
Taux de succès: 100.0%

Détails :
   OK Price: values_to_be_between
   OK Required age: values_to_be_between
   OK Metacritic score: values_to_be_between
   OK Review score: values_to_be_between


In [36]:
# DataFrame récapitulatif pour visualisation
import pandas as pd

rapport = []
for r in result_list:
    exp = r.expectation_config
    exp_type = exp.type.replace("expect_column_", "").replace("expect_", "")
    kwargs = exp.kwargs
    column = kwargs.get('column', kwargs.get('column_A', '-'))

    if 'not_be_null' in exp.type:
        pilier = "Complétude"
    elif 'unique' in exp.type:
        pilier = "Unicité"
    elif 'be_of_type' in exp.type:
        pilier = "Exactitude"
    elif 'be_between' in exp.type:
        if 'year' in column.lower():
            pilier = "Actualité"
        else:
            pilier = "Validité"
    elif 'greater_than' in exp.type or 'pair' in exp.type:
        pilier = "Cohérence"
    else:
        pilier = "Autre"

    rapport.append({
        "Pilier": pilier,
        "Colonne": column,
        "Test": exp_type,
        "Résultat": "Succès" if r.success else "Échec"
    })

df_rapport = pd.DataFrame(rapport)
df_rapport

,Pilier,Colonne,Test,Résultat
0,Validité,Price,values_to_be_between,Succès
1,Validité,Required age,values_to_be_between,Succès
2,Validité,Metacritic score,values_to_be_between,Succès
3,Validité,Review score,values_to_be_between,Succès


## Validation des tables relationnelles Gold

In [ ]:
# Charger les tables relationnelles depuis PostgreSQL
try:
    df_tags = pd.read_sql("SELECT * FROM game_tags", engine)
    df_genres = pd.read_sql("SELECT * FROM game_genres", engine) 
    df_developers = pd.read_sql("SELECT * FROM game_developers", engine)
    df_publishers = pd.read_sql("SELECT * FROM game_publishers", engine)
    print(f"Tags: {len(df_tags):,} lignes")
    print(f"Genres: {len(df_genres):,} lignes") 
    print(f"Developers: {len(df_developers):,} lignes")
    print(f"Publishers: {len(df_publishers):,} lignes")
except Exception as e:
    print(f"Erreur PostgreSQL pour tables relationnelles: {e}")
    # Fallback vers fichiers CSV
    df_tags = pd.read_csv('../data/Gold/game_tags.csv')
    df_genres = pd.read_csv('../data/Gold/game_genres.csv')
    df_developers = pd.read_csv('../data/Gold/game_developers.csv')
    df_publishers = pd.read_csv('../data/Gold/game_publishers.csv')
    print(f"Utilisation des fichiers CSV en fallback")

In [37]:
def validate_relational_table(df_table, table_name, id_col, value_col, main_df):
    """Valide une table relationnelle Gold"""
    results = {"table": table_name, "tests": [], "passed": 0, "failed": 0}

    # Test 1: Pas de null dans la colonne ID
    null_ids = df_table[id_col].isnull().sum()
    passed = null_ids == 0
    results["tests"].append(f"{id_col} not null: {'OK' if passed else 'FAILED'}")
    results["passed" if passed else "failed"] += 1

    # Test 2: Pas de null dans la colonne valeur
    null_vals = df_table[value_col].isnull().sum()
    passed = null_vals == 0
    results["tests"].append(f"{value_col} not null: {'OK' if passed else 'FAIL'}")
    results["passed" if passed else "failed"] += 1

    # Test 3: Pas de valeurs vides
    empty_vals = (df_table[value_col] == '').sum()
    passed = empty_vals == 0
    results["tests"].append(f"{value_col} not empty: {'OK' if passed else 'FAILED'}")
    results["passed" if passed else "failed"] += 1

    # Test 4: tous les AppID existent dans games_gold
    orphan_ids = set(df_table[id_col]) - set(main_df['AppID'])
    passed = len(orphan_ids) == 0
    results["tests"].append(f"Référential integrity: {'OK' if passed else 'FAILED'} ({len(orphan_ids)} orphans)")
    results["passed" if passed else "failed"] += 1

    return results


tables_results = []
tables_results.append(validate_relational_table(df_tags, "game_tags", "AppID", "Tag", df))
tables_results.append(validate_relational_table(df_genres, "game_genres", "AppID", "Genre", df))
tables_results.append(validate_relational_table(df_developers, "game_developers", "AppID", "Developer", df))
tables_results.append(validate_relational_table(df_publishers, "game_publishers", "AppID", "Publisher", df))

for r in tables_results:
    print(f"\n{r['table']}")
    for t in r['tests']:
        print(f"   {t}")
    print(f"   -> {r['passed']}/{r['passed']+r['failed']} tests passés")


game_tags
   AppID not null: OK
   Tag not null: OK
   Tag not empty: OK
   Référential integrity: OK (0 orphans)
   -> 4/4 tests passés

game_genres
   AppID not null: OK
   Genre not null: OK
   Genre not empty: OK
   Référential integrity: OK (0 orphans)
   -> 4/4 tests passés

game_developers
   AppID not null: OK
   Developer not null: OK
   Developer not empty: OK
   Référential integrity: OK (0 orphans)
   -> 4/4 tests passés

game_publishers
   AppID not null: OK
   Publisher not null: OK
   Publisher not empty: OK
   Référential integrity: OK (0 orphans)
   -> 4/4 tests passés


In [33]:
# Résumé global
total_passed = stats['successful_expectations'] + sum(r['passed'] for r in tables_results)
total_tests = stats['evaluated_expectations'] + sum(r['passed']+r['failed'] for r in tables_results)

print(f"\nTable principale (games_gold):")
print(f"   {stats['successful_expectations']}/{stats['evaluated_expectations']} expectations ({stats['success_percent']:.0f}%)")

print(f"\nTables relationnelles:")
for r in tables_results:
    pct = r['passed']/(r['passed']+r['failed'])*100
    print(f"   {r['table']}: {r['passed']}/{r['passed']+r['failed']} ({pct:.0f}%)")


print(f"\nTOTAL: {total_passed}/{total_tests} tests passés ({total_passed/total_tests*100:.0f}%)")



Table principale (games_gold):
   4/4 expectations (100%)

Tables relationnelles:
   game_tags: 4/4 (100%)
   game_genres: 4/4 (100%)
   game_developers: 4/4 (100%)
   game_publishers: 4/4 (100%)

TOTAL: 20/20 tests passés (100%)


In [ ]:
# Résumé par pilier
summary = df_rapport.groupby('Pilier').agg(
    Tests=('Résultat', 'count'),
    Succès=('Résultat', lambda x: (x == 'Succès').sum())
)
summary['Taux'] = (summary['Succès'] / summary['Tests'] * 100).round(1).astype(str) + '%'
print(summary.to_string())
print(f"\nTOTAL: {stats['successful_expectations']}/{stats['evaluated_expectations']} expectations réussies ({stats['success_percent']:.0f}%)")

          Tests  Succès    Taux
Pilier                         
Validité      4       4  100.0%

TOTAL: 4/4 expectations réussies (100%)


## Récapitulatif des 13 Expectations

| Pilier | Expectation | Colonne |
|--------|-------------|---------|
| Complétude | not_be_null 100% | AppID |
| Complétude | not_be_null 100% | Name |
| Complétude | not_be_null 95% | Release date |
| Unicité | be_unique | AppID |
| Exactitude | be_of_type float64 | Price |
| Exactitude | be_of_type int64 | AppID |
| Validité | be_between 0-1000 | Price |
| Validité | be_between 0-21 | Required age |
| Validité | be_between 0-100 | Metacritic score |
| Validité | be_between 0-100 | Review score |
| Cohérence | A >= B | Owners_max vs Owners_min |
| Cohérence | A >= B | Total reviews vs Positive |
| Actualité | be_between 1980-2026 | Release year |